# Image Captioning with LLaVA 1.5

This notebook isolates the image-to-text generation component from the original project archive. It uses the Hugging Face model `llava-hf/llava-1.5-7b-hf` with optional 4-bit quantization.

The archive also contained a commented BLIP example; the active captioning flow was based on LLaVA.

In [ ]:
import torch
from PIL import Image
from pathlib import Path
from transformers import BitsAndBytesConfig, pipeline

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
IMAGE_PATH = "sample_image.jpg"

if not torch.cuda.is_available():
    print("CUDA is not available. Loading the 7B model may be too slow or may run out of memory.")

## 1. Configure quantization and load the model

4-bit loading requires a compatible CUDA environment and bitsandbytes installation.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

captioner = pipeline(
    task="image-to-text",
    model=MODEL_ID,
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto"
)

## 2. Generate an image description

The prompt asks the model to describe the main object, its color, and relevant visual details while ignoring the background.

In [ ]:
def describe_image(image_path: str, max_new_tokens: int = 200) -> str:
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    image = Image.open(path).convert("RGB")
    prompt = (
        "USER: <image>\n"
        "Describe the main object in the image, including its color and "
        "important details. Focus on the object and ignore the background.\n"
        "ASSISTANT:"
    )

    result = captioner(
        image,
        prompt=prompt,
        generate_kwargs={"max_new_tokens": max_new_tokens}
    )

    generated_text = result[0]["generated_text"]
    if "ASSISTANT:" in generated_text:
        generated_text = generated_text.split("ASSISTANT:", 1)[1]

    return generated_text.strip()

In [ ]:
if Path(IMAGE_PATH).is_file():
    description = describe_image(IMAGE_PATH)
    print("Image description:")
    print(description)
else:
    print(f"Add an image at {IMAGE_PATH} to generate a caption.")

## Optional lightweight baseline: BLIP

The original archive included this commented baseline. It can be used as a separate comparison, without loading the 7B LLaVA model:

```python
from transformers import pipeline

blip_captioner = pipeline(
    "image-to-text",
    model="Salesforce/blip-image-captioning-base"
)
print(blip_captioner("sample_image.jpg"))
```

## Limitations

- Caption quality depends on image content, model behavior, and prompt wording.
- LLaVA 1.5 7B is compute-intensive; 4-bit quantization reduces memory use but does not eliminate hardware requirements.
- Review generated descriptions before using them in a lost-and-found decision or public listing.